# Simulations (Supp Fig S1, Supp Tables S1-S2)

This notebook covers the simulation results:
1. the **calibrated simulator** (`accumap.realistic_sim`), checked against the experimental data (**Supp Table S1**);
2. the **benchmark** of UMAP, similarity-weighted multidimensional scaling (the original accumap) and accumap on calibrated, real-layout, idealized and single-component simulations (**Supp Fig S1, Supp Table S2**), including downstream spatial domains from Concordex.

**Runtime.** By default (`RECOMPUTE = False`) the notebook simulates one array live (CPU, a few minutes), reconstructs it with UMAP and, if a CUDA GPU is available, with accumap, and draws every figure and table from the cached results in `paper_results/simulations/`. `RECOMPUTE = True` reruns the whole benchmark (21 arrays and all methods; about 12 hours on one GPU, most of it similarity MDS with the full objective, which needs the baseline's Rust solver: see `baselines/__init__.py`) and writes to `paper_results/simulations_recomputed/`.

**Data.** Table S1 compares with replicate c58_8 of Hu et al. (2025) and the real-layout simulation uses its bead positions; both need the c58_8 files from [SCP2577](https://singlecell.broadinstitute.org/single_cell/study/SCP2577/computational-reconstruction-for-imaging-free-spatial-transcriptomics) in the repository's `data/` folder. Concordex needs the `concordex` package and the Slide-seq cerebellum section (`data/cerebellum_processed.h5ad`).

In [ ]:
import os, re, json, glob, colorsys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial import distance_matrix
from scipy.spatial.distance import pdist
from accumap.realistic_sim import simulate_realistic
from accumap.metrics import evaluate
from accumap.real_data import load_hu, DATA_DIR

RECOMPUTE = False                      # True: rerun the full benchmark (GPU, about 12 h)
RESULTS = "paper_results"
SIMDIR = os.path.join(RESULTS, "simulations_recomputed" if RECOMPUTE else "simulations")
FIGURES = "../final_figures"
os.makedirs(FIGURES, exist_ok=True); os.makedirs(SIMDIR, exist_ok=True)

try:
    import torch
    GPU = torch.cuda.is_available()
except ImportError:
    GPU = False
HAVE_C58_8 = os.path.exists(os.path.join(DATA_DIR, "c58_8_matched_bead_location.csv.gz"))
print(f"GPU: {GPU}, c58_8 data: {HAVE_C58_8}, RECOMPUTE: {RECOMPUTE}")


def umap_embed(C, seed):
    """UMAP of the receivers as used throughout the paper: log(1 + C), cosine metric, 25 neighbors, minimum distance 0.99."""
    import umap
    E = umap.UMAP(random_state=seed, metric="cosine", n_neighbors=25, min_dist=0.99, n_components=2,
                  n_epochs=400).fit_transform(np.log1p(C))
    E = E - E.mean(0)
    return E * 3000.0 / np.max(distance_matrix(E, E))


def fmt(m):   # metrics dict -> the compact string used in the cached results
    return f"recv p50 {m['median']:.1f} p90 {m['p90']:.1f} pairRMSE {m['pairwise_rmse']:.1f} J@10 {m['jaccard10']:.3f}"

## 1. The calibrated simulator

`simulate_realistic` places beads uniformly in a 3 mm disc with replicate c58_8's bead numbers and draws counts from the Poisson model fit to c58_8 at its imaging positions, with each component at its fitted level: the contact-dominated kernel, a background of mislocalized transfer, per-bead capture and abundance, count overdispersion and barcode collisions (main-text Methods; Supplementary Note 1). Each component can be switched off; the idealized simulation switches all of them off and uses a Gaussian kernel.

In [ ]:
C, X, U = simulate_realistic(seed=0)
N = C.sum(1)
print(f"{C.shape[0]} receivers x {C.shape[1]} senders; median {np.median(N):.0f} UMIs per receiver, "
      f"{np.median((C > 0).sum(1)):.0f} senders detected; top sender carries {np.median(C.max(1) / N):.0%} of a receiver's UMIs")

## 2. Supp Table S1: does the simulation look like the experimental data?

Per-receiver count statistics for c58_8 and one array of each simulation, together with UMAP's error. The idealized simulation is far less contact-dominated than the data; the calibrated one matches it.

In [ ]:
def split_frac(C, P_other, k=5, span=200.0):
    """Fraction of receivers whose k highest-count partners span more than `span` um (a signature of collisions)."""
    top = np.argsort(-C, axis=1)[:, :k]
    ok = (C[np.arange(len(C))[:, None], top] > 0).all(1)
    return (np.array([pdist(P_other[t]).max() for t in top[ok]]) > span).mean()


def links_300(C, X, k=15):
    """Fraction of 15-nearest-neighbor cosine-similarity links spanning more than 300 um."""
    L = np.log1p(C); L /= np.linalg.norm(L, axis=1, keepdims=True) + 1e-12
    far = n = 0
    for i in range(0, len(L), 1000):
        S = L[i:i + 1000] @ L.T
        S[np.arange(len(S)), np.arange(i, i + len(S))] = -1
        nb = np.argpartition(-S, k, axis=1)[:, :k]
        d = np.linalg.norm(X[nb] - X[i:i + 1000, None, :], axis=2)
        far += (d > 300).sum(); n += d.size
    return far / n


def table_row(C, X, U):
    N = C.sum(1); f30 = []
    for i in range(0, len(C), 1000):
        d = np.sqrt(((X[i:i + 1000, None, :] - U[None]) ** 2).sum(-1))
        f30.append((C[i:i + 1000] * (d < 30)).sum(1) / np.maximum(N[i:i + 1000], 1))
    r = {"umi": float(np.median(N)), "detected": float(np.median((C > 0).sum(1))),
         "top1": float(np.median(C.max(1) / np.maximum(N, 1))), "f30": float(np.median(np.concatenate(f30))),
         "split_receivers": float(split_frac(C, U)), "links_300": float(links_300(C, X))}
    for s in (0, 1):
        r[f"umap_seed{s}"] = fmt(evaluate(umap_embed(C, s), X))
    return r


IDEAL = dict(kernel="gaussian", background=False, heterogeneity=False, overdispersion=False, collisions=0.0)
if RECOMPUTE:
    Cr, Xr, Ur = load_hu("c58_8", count="umi")
    t1 = {"real c58_8": table_row(Cr.astype(np.float32), Xr, Ur),
          "calibrated, uniform disc": table_row(*simulate_realistic(seed=0)),
          "calibrated, c58_8 positions": table_row(*simulate_realistic(seed=0, receivers=Xr, senders=Ur)),
          "idealized": table_row(*simulate_realistic(seed=0, **IDEAL))}
    json.dump(t1, open(os.path.join(SIMDIR, "table_s1.json"), "w"), indent=1)
t1 = json.load(open(os.path.join(SIMDIR, "table_s1.json")))
cols = {"umi": "median UMIs", "detected": "senders detected", "top1": "top sender share", "f30": "UMIs < 30 um",
        "split_receivers": "split receivers", "links_300": "links > 300 um", "umap_seed0": "UMAP (seed 0)"}
pd.DataFrame(t1).T[list(cols)].rename(columns=cols)

## 3. The benchmark

Each simulated array is reconstructed by UMAP, by similarity MDS (similarity-weighted multidimensional scaling) over all bead pairs and restricted to each bead's five most similar neighbors (the original accumap, Supplementary Note 3; its similarity-to-distance curve is calibrated on 100 beads with known positions, an advantage the other methods do not get), and by accumap. Below, one calibrated array is reconstructed live; the full benchmark runs only with `RECOMPUTE = True`.

In [ ]:
from baselines.simulation import cos_sim
from baselines.curves import fit_curve, build_knn_mask, accumap_solve


def run_methods(C, X, seed, methods=("umap", "old_full", "old_knn", "new")):
    rec = {"umap": umap_embed(C, seed)}
    if "old_full" in methods or "old_knn" in methods:
        S = cos_sim(C); np.fill_diagonal(S, 0.0)
        D = fit_curve(S, distance_matrix(X, X))          # calibration: 100 beads with known distances
        if "old_full" in methods:
            rec["old_full"] = accumap_solve(rec["umap"], D, S, cycles=10)
        if "old_knn" in methods:
            rec["old_knn"] = accumap_solve(rec["umap"], D, np.where(build_knn_mask(S, 5), S, 0.0), cycles=10)
    if "new" in methods:
        from accumap.count_model import reconstruct
        rec["new"] = reconstruct(C, rec["umap"], array_diameter=3000.0, verbose=False)["receivers"]
    return rec


live = run_methods(C, X, seed=0, methods=("umap", "new") if GPU else ("umap",))
for m, P in live.items():
    print(f"{m:8s}", {k: round(v, 3) for k, v in evaluate(P, X).items()})
if not GPU:
    print("accumap needs a CUDA GPU; cached results are used below.")

In [ ]:
DATASETS = {"realistic": range(5), "reallayout": range(3), "idealized": range(3),
            **{f"+{c}": range(2) for c in ("kernel", "background", "heterogeneity", "overdispersion", "collisions")}}
FULL_OBJECTIVE = {("realistic", 0), ("realistic", 1), *{("idealized", s) for s in range(3)},
                  *{(f"+{c}", 0) for c in ("kernel", "background", "heterogeneity", "overdispersion", "collisions")}}


def make(dataset, seed):
    if dataset == "realistic":
        return simulate_realistic(seed=seed)
    if dataset == "reallayout":
        _, Xr, Ur = load_hu("c58_8", count="umi")
        return simulate_realistic(seed=seed, receivers=Xr, senders=Ur)
    if dataset == "idealized":
        return simulate_realistic(seed=seed, **IDEAL)
    comp = dataset[1:]   # idealized plus one calibrated component
    return simulate_realistic(seed=seed, **{**IDEAL, comp: {"kernel": "fitted", "overdispersion": "fitted", "collisions": 0.006}.get(comp, True)})


if RECOMPUTE:
    for ds, seeds in DATASETS.items():
        for s in seeds:
            C_, X_, U_ = make(ds, s)
            methods = ("umap", "old_full", "old_knn", "new") if (ds, s) in FULL_OBJECTIVE else ("umap", "old_knn", "new")
            rec = run_methods(C_, X_, s, methods)
            json.dump({m: fmt(evaluate(P, X_)) for m, P in rec.items()}, open(os.path.join(SIMDIR, f"{ds}_seed{s}.json"), "w"))
            np.savez_compressed(os.path.join(SIMDIR, f"{ds}_seed{s}.npz"), X=X_, U=U_, **rec)
            if (ds, s) == ("realistic", 0):
                np.savez_compressed(os.path.join(SIMDIR, "realistic_seed0_example.npz"), X=X_, U=U_, **rec)
            print(ds, s, "done", flush=True)

### Downstream spatial domains (Concordex)

Each simulated receiver within 15 µm of a bead of a Slide-seq cerebellum section is given that bead's expression; Concordex spatial domains (k-means, k = 4) are computed from the true and from each reconstructed position, and agreement is the adjusted Rand index. Recomputed only with `RECOMPUTE = True` (needs `concordex` and the cerebellum data).

In [ ]:
if RECOMPUTE:
    import anndata as ad, scanpy as sc
    from scipy.spatial import cKDTree
    from sklearn.cluster import KMeans
    from sklearn.metrics import adjusted_rand_score
    from concordex.tools import calculate_concordex

    def domains(A, key):
        calculate_concordex(A, "X_pca", use_rep=key, compute_similarity=True, key_added=key, recompute_neighbors=True)
        return KMeans(n_clusters=4, n_init=20, random_state=42).fit_predict(A.obsm[f"{key}_nbc"])

    cb = sc.read_h5ad(os.path.join(DATA_DIR, "cerebellum_processed.h5ad"))
    T = cb.obsm["spatial"] * 0.65; T = T - T.mean(0); tree = cKDTree(T)   # 0.65 um per pixel
    for f in sorted(glob.glob(os.path.join(SIMDIR, "*_seed[0-9].npz"))):
        Z = np.load(f); d, j = tree.query(Z["X"]); keep = d < 15.0
        A = ad.AnnData(obs=cb.obs.iloc[j[keep]].reset_index(drop=True))
        A.obsm["X_pca"] = cb.obsm["X_pca"][j[keep]]; A.obsm["truth"] = Z["X"][keep]
        truth = domains(A, "truth"); res = {}
        for m in ("umap", "old_full", "old_knn", "new"):
            if m in Z.files:
                A.obsm[m] = np.nan_to_num(Z[m][keep]); res[m] = adjusted_rand_score(truth, domains(A, m))
        json.dump(res, open(f.replace(".npz", "_concordex.json"), "w"))

## 4. Supp Fig S1

(A-E) One calibrated array, shown at the truth and as reconstructed by each method; receivers are teal if they keep at least 9 of their 10 true nearest neighbors in that layout, black otherwise. (F-H) Median position error, neighborhood recovery and Concordex domain agreement on the idealized and calibrated simulations, one point per array.

In [ ]:
TITLE_FS, LABEL_FS, TICK_FS, LETTER_FS = 13, 12, 10, 18
METHODS = [("umap", "UMAP", "#dc143c"), ("old_full", "similarity MDS (full)", "#eda100"),
           ("old_knn", "similarity MDS (5-NN)", "#1baf7a"), ("new", "accumap", "#4169e1")]
PLOT_DATASETS = [("idealized", "idealized\nsimulation"), ("realistic", "calibrated\nsimulation")]
REPS = ("c58_8", "c58_4", "c58_5", "c58_16")
MU = "$\\mu$m"


def load_sim():
    """(dataset, method) -> {seed: (median, p90, pairRMSE, J10)} and (dataset, method) -> {seed: ARI}."""
    met, ari = {}, {}
    for f in glob.glob(os.path.join(SIMDIR, "*_seed*.json")):
        base = os.path.basename(f)
        ds, seed = re.match(r"(.+)_seed(\d+)", base).groups()
        for m, v in json.load(open(f)).items():
            if base.endswith("_concordex.json"):
                if m != "n_receivers_in_tissue":
                    ari.setdefault((ds, m), {})[int(seed)] = v
            else:
                met.setdefault((ds, m), {})[int(seed)] = [float(x) for x in re.findall(r"(?:p50|p90|pairRMSE|J@10) ([\d.]+)", v)]
    return met, ari


def style(ax):
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    ax.tick_params(axis="both", labelsize=TICK_FS)


def letter(ax, s, dx=-0.2, dy=1.04):
    ax.text(dx, dy, s, transform=ax.transAxes, fontsize=LETTER_FS, fontweight="bold", va="bottom")


def dotpanel(ax, get, ylabel, log=False):
    w = 0.18
    for gi, (ds, _) in enumerate(PLOT_DATASETS):
        for mi, (m, lab, col) in enumerate(METHODS):
            vals = get(ds, m)
            if not vals:
                continue
            x = gi + (mi - 1.5) * w
            ax.scatter(np.full(len(vals), x), vals, s=30, color=col, edgecolors="white", linewidths=0.8, zorder=3,
                       label=lab if gi == 1 else None)
            ax.plot([x - w * 0.4, x + w * 0.4], [np.mean(vals)] * 2, color=col, lw=2.2, zorder=2)
    ax.set_xticks(range(len(PLOT_DATASETS))); ax.set_xticklabels([d[1] for d in PLOT_DATASETS], fontsize=TICK_FS)
    ax.set_xlim(-0.55, len(PLOT_DATASETS) - 0.45)
    ax.set_ylabel(ylabel, fontsize=LABEL_FS)
    if log:
        ax.set_yscale("log")
        ax.set_yticks([5, 10, 20, 50, 100]); ax.set_yticklabels(["5", "10", "20", "50", "100"]); ax.minorticks_off()
    style(ax)

In [ ]:
met, ari = load_sim()

fig = plt.figure(figsize=(20, 10.5))
gs = fig.add_gridspec(2, 1, height_ratios=[0.95, 1.0], hspace=0.25)
bot = gs[1].subgridspec(1, 3, wspace=0.32)   # metrics (second row)
top = gs[0].subgridspec(1, 5, wspace=0.05)   # examples (first row)

ax = fig.add_subplot(bot[0])
dotpanel(ax, lambda ds, m: [v[0] for v in met.get((ds, m), {}).values()], f"Median per-bead error ({MU})  (← better)", log=True)
ax.set_ylim(3.5, 300)   # headroom for the legend
ax.set_title("Position error", fontsize=TITLE_FS); letter(ax, "F")
ax.legend(fontsize=TICK_FS, frameon=False, loc="upper left", ncol=2, handletextpad=0.2, columnspacing=0.8)
ax = fig.add_subplot(bot[1])
dotpanel(ax, lambda ds, m: [v[3] for v in met.get((ds, m), {}).values()], "10-NN Jaccard distance  (← better)")
ax.set_title("Neighborhood recovery", fontsize=TITLE_FS); letter(ax, "G")
ax = fig.add_subplot(bot[2])
dotpanel(ax, lambda ds, m: list(ari.get((ds, m), {}).values()), "Adjusted Rand index  (better →)")
ax.set_ylim(0, 1)
ax.set_title("Downstream domains (Concordex)", fontsize=TITLE_FS); letter(ax, "H")

# D-H: example reconstructions of calibrated array seed 0 (accumap: final method)
Z = np.load(os.path.join(SIMDIR, "realistic_seed0_example.npz"))
X = Z["X"]
from scipy.spatial import cKDTree
from matplotlib.patches import Patch
GREEN, BLACK = "#5fd8a4", "#1a1a1a"
true_nn = cKDTree(X).query(X, 11)[1][:, 1:]
order = np.random.default_rng(0).permutation(len(X))      # random drawing order: neither color hides the other


def align(P, T):
    Pc, Tc = P - P.mean(0), T - T.mean(0)
    U, s, Vt = np.linalg.svd(Pc.T @ Tc)
    return Pc @ (U @ Vt) * (s.sum() / np.sum(Pc ** 2)), Tc


examples = [("ground truth\n(calibrated sim)", None), ("UMAP", Z["umap"]), ("similarity MDS (full)", Z["old_full"]),
            ("similarity MDS (5-NN)", Z["old_knn"]), ("accumap", Z["new"])]
for k, (name, R) in enumerate(examples):
    ax = fig.add_subplot(top[k])
    if R is None:
        P = X - X.mean(0); kept = np.full(len(X), 10)
    else:
        P, Tc = align(np.nan_to_num(R), X)
        name += f"\nmedian error {np.median(np.linalg.norm(P - Tc, axis=1)):.1f} {MU}"
        rec_nn = cKDTree(P).query(P, 11)[1][:, 1:]
        kept = np.array([len(set(x) & set(y)) for x, y in zip(true_nn, rec_nn)])
    col = np.where(kept >= 9, GREEN, BLACK)
    ax.scatter(P[order, 0], P[order, 1], c=col[order], s=1.5, linewidths=0, rasterized=True)
    if R is None:
        ax.legend(handles=[Patch(facecolor=GREEN, label="\u2265 9 of 10 true neighbors kept"), Patch(facecolor=BLACK, label="fewer")],
                  fontsize=TICK_FS, frameon=False, loc="upper center", bbox_to_anchor=(0.5, 0.02))
    ax.set_aspect("equal"); ax.axis("off")
    ax.set_xlim(-1750, 1750); ax.set_ylim(-1750, 1750)
    ax.set_title(name, fontsize=TITLE_FS)
    ax.text(0.0, 1.02, "ABCDE"[k], transform=ax.transAxes, fontsize=LETTER_FS, fontweight="bold", va="bottom")
fig.savefig(os.path.join(FIGURES, "FigS1_final.png"), dpi=250, bbox_inches="tight")
plt.show()

## 5. Supp Table S2

Mean and SD over simulated arrays for every dataset and method; "better than UMAP" counts arrays.

In [ ]:
parse = lambda m: [float(x) for x in re.findall(r"(?:p50|p90|pairRMSE|J@10) ([\d.]+)", m)]
met, ari = {}, {}
for f in glob.glob(os.path.join(SIMDIR, "*_seed*.json")):
    base = os.path.basename(f)
    if not re.match(r".+_seed\d+", base):
        continue
    ds, seed = re.match(r"(.+)_seed(\d+)", base).groups()
    for m, v in json.load(open(f)).items():
        if m == "n_receivers_in_tissue" or m in ("new_v1", "new_warp"):
            continue
        (ari if base.endswith("_concordex.json") else met).setdefault((ds, m), {})[int(seed)] = v if isinstance(v, float) else parse(v)
NAMES = {"umap": "UMAP", "old_full": "similarity MDS (full)", "old_knn": "similarity MDS (5-NN)", "new": "accumap"}
rows = []
for ds in DATASETS:
    for m in NAMES:
        if (ds, m) not in met:
            continue
        A = np.array([met[(ds, m)][s] for s in sorted(met[(ds, m)])])
        r = {"dataset": ds, "method": NAMES[m], "n": len(A)}
        for i, k in enumerate(("median (um)", "p90 (um)", "pairwise RMSE (um)", "10-NN Jaccard")):
            r[k] = f"{A[:, i].mean():.3g} ± {A[:, i].std(ddof=1):.2g}" if len(A) > 1 else f"{A[0, i]:.3g}"
        if (ds, m) in ari:
            a = np.array(list(ari[(ds, m)].values())); r["Concordex ARI"] = f"{a.mean():.2f} ({a.min():.2f}-{a.max():.2f})"
        rows.append(r)
pd.DataFrame(rows).set_index(["dataset", "method"])